# What does one of our training runs cost on *this* machine?

CSED 505 · panel 5 · [WashingtonCsed504](https://github.com/TrueRottweiler/WashingtonCsed504)

The poster says a training run is 1.024 billion tokens and ninety minutes. **Ninety minutes on
what** is the question anyone reading it will actually ask, because almost nobody owns two
Blackwell cards. This notebook measures the same two models here, on whatever you are sitting
in front of, and converts the answer into the two numbers that decide whether a student could
attempt a study like this:

- **how long one run takes**, and
- **whether the model fits in memory at all** — which is a legitimate answer, not a failure.

It needs no corpus, no tokenizer and no data from the repository. The token stream is random
integers, which is useless for learning and *identical for timing*, because a transformer's cost
per step does not depend on which token ids arrive.

**Runtime → Change runtime type** first, and note which accelerator you picked. Then run all
three cells (about two minutes) and send back what the last one prints.

In [ ]:
# Colab already has torch; transformers is usually there too but pin nothing, because the
# point of this measurement is the machine as a student would find it.
%pip install --quiet transformers

!wget -q -O bench_portable.py https://raw.githubusercontent.com/TrueRottweiler/WashingtonCsed504/main/src/a2-nlp/bench_portable.py
print('benchmark downloaded')

In [ ]:
# 40 timed steps after 8 warmup. Warmup is not optional -- the first steps pay for kernel
# autotuning and allocator growth, and on a cold card a short benchmark reads about 10% high.
!python bench_portable.py --steps 40 --warmup 8 --out bench_result.json

In [ ]:
# The block to send back. Everything here is read from the machine rather than typed, so the
# poster's hardware table cannot end up describing a machine nobody ran on.
import json, platform, subprocess

import torch

info = {
    'label': 'CHANGE ME -- e.g. "Colab free tier" or "MacBook Pro M3 Max 36GB"',
    'platform': platform.platform(),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'total_vram_gb': (round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
                      if torch.cuda.is_available() else None),
    'mps': bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()),
    'results': json.load(open('bench_result.json')),
}

# Edit the label above, re-run this cell, then copy everything below the line.
print('-' * 70)
print(json.dumps(info, indent=2))

## What the numbers mean

| field | reading it |
|---|---|
| `tokens_per_s` | sustained training throughput, forward + backward + optimizer step |
| `peak_gb` | high-water GPU memory at batch 128 × 128 tokens |
| `full_run_hours` | one 62,500-step run — what the study's standard cell costs here |
| `vs_workstation` | how many times slower than one RTX PRO 6000 Blackwell Max-Q |
| `project_hours_here` | what the whole 83-GPU-hour A2 project would have cost on this machine |
| `error: out of memory` | **a result.** The model does not fit at this batch, and knowing that before planning a term is worth more than any throughput figure |

Two warnings the script prints for itself, both of which we earned:

- **Contention.** If another process holds the same card, the numbers read low — on our own
  workstation a contended measurement came out at *half* the sustained rate, and the first version
  of the check missed it because it only looked at its own process.
- **Precision.** CUDA runs in bf16 or fp16; Apple MPS and CPU are left in fp32 deliberately, since
  autocast there is unsupported or slower. **The MPS row is therefore not directly comparable to
  the CUDA rows**, and the poster labels it that way rather than quietly putting them side by side.